# Export sample planes (preprocessed + binary plaques)

Pull **20 sample planes** out of a sample's preprocessed (deconvolved) image together with the
**same region of the binary Cx43 plaque mask**, and package them for download.

For each selected plane it exports:
- the preprocessed grayscale plane (region),
- the same region with the binary plaques overlaid (red contour),

and it also writes the raw 20-plane sub-stacks as TIFFs (openable in ImageJ / napari). Everything is
zipped into a single archive you can download.

Configure the sample, the plane selection, and an optional crop region in the **Config** cell.

In [ ]:
from pathlib import Path
import sys
import shutil

import numpy as np
import matplotlib.pyplot as plt
from skimage import io, img_as_ubyte

BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))              # run_pipeline (repo root)
sys.path.append(str(BASE_DIR / 'src'))

from run_pipeline import resolve_config, OUTPUTS, _crop_seg

RESULTS_DIR = BASE_DIR / 'results' / 'exported_planes'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Config

In [ ]:
SAMPLE = 'sample_1'                       # which sample (must exist in run_pipeline.SAMPLES)

cfg = resolve_config(SAMPLE)
# default to the sample's pipeline outputs; edit if your files live elsewhere
# (e.g. sample_1's deconvolved may be at data/preprocessed/...):
PREPROCESSED_PATH = BASE_DIR / cfg['out_dir'] / OUTPUTS['deconvolved']
BINARY_PATH       = BASE_DIR / cfg['out_dir'] / OUTPUTS['binary_mask']

N_PLANES = 20                             # how many planes to export
PLANES   = None                           # None -> N_PLANES evenly spaced across the stack;
                                          # or give explicit indices, e.g. list(range(140, 160))
CROP_Y   = None                           # (y0, y1) region, or None for the full height
CROP_X   = None                           # (x0, x1) region, or None for the full width

DISP_VMIN = 0                             # grayscale display range for the PNGs
DISP_VMAX = None                          # None -> robust 99.5th percentile

print('sample     :', SAMPLE)
print('preprocessed:', PREPROCESSED_PATH)
print('binary      :', BINARY_PATH)

## Load and align the two stacks

In [ ]:
binary = io.imread(BINARY_PATH) > 0
deconv = io.imread(PREPROCESSED_PATH)

# the binary mask lives in the segmentation frame; crop the preprocessed image to match if needed
if deconv.shape != binary.shape:
    deconv = _crop_seg(deconv, cfg)
assert deconv.shape == binary.shape, f"shape mismatch: {deconv.shape} vs {binary.shape}"
print('aligned stack shape:', deconv.shape)

## Select planes + region

In [ ]:
Z = deconv.shape[0]
if PLANES is None:
    planes = np.unique(np.linspace(0, Z - 1, N_PLANES).round().astype(int))
else:
    planes = np.asarray(sorted(set(int(p) for p in PLANES if 0 <= p < Z)))
print(f"{len(planes)} planes: {planes.tolist()}")

ys = slice(*CROP_Y) if CROP_Y else slice(None)
xs = slice(*CROP_X) if CROP_X else slice(None)

sub_deconv = deconv[planes][:, ys, xs]
sub_binary = binary[planes][:, ys, xs]
vmax = DISP_VMAX if DISP_VMAX is not None else float(np.percentile(sub_deconv, 99.5))
print('exported region shape:', sub_deconv.shape, '| display vmax =', round(vmax, 1))

## Preview grid (preprocessed + red plaque contour)

In [ ]:
ncol = 5
nrow = int(np.ceil(len(planes) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3 * ncol, 3 * nrow), squeeze=False)
for ax in axes.ravel():
    ax.axis('off')
for i, p in enumerate(planes):
    ax = axes[i // ncol, i % ncol]
    ax.imshow(sub_deconv[i], cmap='gray', vmin=DISP_VMIN, vmax=vmax)
    ax.contour(sub_binary[i], levels=[0.5], colors='red', linewidths=0.4)
    ax.set_title(f'plane {p}', fontsize=9)
fig.suptitle(f'{SAMPLE}: preprocessed + binary plaques (red)')
plt.tight_layout()
fig.savefig(RESULTS_DIR / f'{SAMPLE}_preview_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## Export files + zip for download

In [ ]:
out_dir = RESULTS_DIR / f'{SAMPLE}_planes'
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True)

# raw 20-plane sub-stacks (openable in ImageJ / napari)
io.imsave(out_dir / 'preprocessed_planes.tif', sub_deconv)
io.imsave(out_dir / 'binary_planes.tif', img_as_ubyte(sub_binary))

# per-plane PNGs: preprocessed | same region with binary plaques (red overlay)
for i, p in enumerate(planes):
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(sub_deconv[i], cmap='gray', vmin=DISP_VMIN, vmax=vmax)
    ax[0].set_title(f'preprocessed (plane {p})')
    ax[1].imshow(sub_deconv[i], cmap='gray', vmin=DISP_VMIN, vmax=vmax)
    ax[1].contour(sub_binary[i], levels=[0.5], colors='red', linewidths=0.5)
    ax[1].set_title('binary plaques (red)')
    for a in ax:
        a.axis('off')
    fig.tight_layout()
    fig.savefig(out_dir / f'plane_{p:03d}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

shutil.copy(RESULTS_DIR / f'{SAMPLE}_preview_grid.png', out_dir / 'preview_grid.png')

# zip everything -> single download
archive = shutil.make_archive(str(RESULTS_DIR / f'{SAMPLE}_planes'), 'zip', out_dir)
print('files written to:', out_dir)
print('download this archive:', archive)